# Joseki library · remote-GPU batch AI local evaluation (KataGo)

Moves the slowest step of the local pipeline (`analyze2.py`, 30~40 minutes per group on the integrated GPU) onto a machine with a discrete GPU and runs it as a batch. The output is `g_<name>.a2.json` in **exactly the same format** as `analyze2.py`.

**How to use (Colab free tier / Kaggle)**
1. Select a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU*; Kaggle: *Settings → Accelerator → GPU*, and turn Internet on).
2. On the local machine, zip the `g_*.n.json` files you want analyzed (see README.md) and upload the zip in the 4th code cell (or put it on Google Drive / a Kaggle Dataset).
3. The weights `kata1-tf3-b11c768-s12752M-d6531M.bin.gz` (about 262MB) are the same file the local machine uses; the easiest way is to upload them once to Drive / a Kaggle Dataset.
4. Run the cells from top to bottom. If the connection drops, reconnect and **re-run all cells**; finished groups are skipped automatically (progress survives only if the output is on Drive or in `/kaggle/working`).
5. The last cell zips the results. After downloading, put them back into `D:\tmp\work` (replacing/adding `g_<name>.a2.json`), then continue with the usual `regen.py` + `tools\build.ps1`.

**Terms of use (must read)**: the Colab free tier may **only be used as an interactive notebook**. This notebook only runs a batch job inside the notebook process; it opens no ports and uses no SSH, tunnels or remote desktop.
**Do not** add a keep-alive script, an ngrok/cloudflared tunnel, or use Colab as a server for the web page to connect to in real time -- the Colab free-tier FAQ explicitly prohibits these, and you may be throttled or banned.
Keep the browser tab open and stay nearby; if it says the usage limit has been reached, stop and come back another day or switch platform instead of working around it.

In [ ]:
import os, subprocess, sys, glob, json, time
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules
WORK = '/kaggle/working' if ON_KAGGLE else ('/content' if ON_COLAB else os.getcwd())
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'other', 'working directory:', WORK)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or '!! No GPU detected: switch to a GPU runtime first')

The next two cells write the scripts into the working directory (same content as the files of the same name in the `remote/` directory).

In [ ]:
%%writefile analyze_remote.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Batch offline AI local evaluation (remote GPU version) -- a line-by-line port of the local pipeline's D:\\tmp\\work\\analyze2.py.

Input : g_<name>.n.json  (output of build_tree in the local pipeline; format as in analyze2.py)
Output: g_<name>.a2.json (exactly the same format as analyze2.py: {node id: {turn,score,wr,cand:[{m,v,s,wr,p,pv,(f)}]}};
                          score/winrate are already converted to the side to move's perspective, so the file can be put
                          straight back into D:\\tmp\\work for merge/regen)
        g_<name>.a2.meta.json (side note: model / KataGo version / visits / timing; does not affect the a2.json format)

What must stay identical to analyze2.py (do not change it, or old and new results are no longer comparable):
  * KataGo config has reportAnalysisWinratesAs=BLACK (also forced once with -override-config on the command line), so all
    score/winrate values are from Black's perspective; they are converted to the side to move (sg=+1 for B, -1 for W)
    before sorting and before computing losses;
  * region = 3%~97% percentile of all stone coordinates in the group + 2 lines of margin, snapped to the edge when close;
    allowMoves is restricted to the region;
  * only "branch points" (>=2 live children) and setup roots (with exactly 1 child) are analyzed;
  * phase 1: main search VB (default 220 visits); when there are stones outside the region, one more control search
    without the outside stones (min(VB,200));
  * phase 2: every move the teacher played that the main search gave no reliable visit count (<max(10, 10% of max visits))
    is evaluated separately and forced (VF, default 100); the result is negated (opponent's perspective -> side to move)
    and marked f=1, p=-1;
  * rules=japanese, komi=6.5, 19x19, untilDepth=40.

Usage:
  python analyze_remote.py --in ./in --out ./out --katago ./katago --model ./model.bin.gz
  (an existing a2.json is skipped automatically => after a disconnect/timeout just run again to resume; --force recomputes)
"""
import argparse, glob, json, os, re, shlex, subprocess, sys, time

L = 'ABCDEFGHJKLMNOPQRST'


def g(c, r):
    return L[c] + str(19 - r)


# --------------------------------------------------------------------------- query construction for one group
class Group:
    """Mirrors the module-level functions/variables of analyze2.py, wrapped in an object so several groups can be looped over."""

    def __init__(self, N):
        self.N = N
        self.R = self.region()
        R = self.R
        self.allow = [g(c, r) for c in range(R['x0'], R['x1'] + 1) for r in range(R['y0'], R['y1'] + 1)]

    @staticmethod
    def parse(pos):
        d = {}
        for t in re.findall(r'[BW]\d+', pos):
            k = int(t[1:])
            d[(k % 18, k // 18)] = t[0]
        return d

    def region(self):  # identical to analyze2.region, line by line
        N = self.N
        cs = set()
        for n in N:
            if n['dead']:
                continue
            if n['mv'] is not None:
                cs.add((n['mv'][0] % 18, n['mv'][0] // 18))
            elif n['par'] == 0:
                for t in re.findall(r'[BW]\d+', n['pos']):
                    k = int(t[1:])
                    cs.add((k % 18, k // 18))
        xs = sorted(c for c, _ in cs)
        ys = sorted(r for _, r in cs)
        pc = lambda a, p: a[min(len(a) - 1, int(len(a) * p))]
        x0 = max(0, pc(xs, .03) - 2)
        x1 = min(17, pc(xs, .97) + 2)
        y0 = max(0, pc(ys, .03) - 2)
        y1 = min(18, pc(ys, .97) + 2)
        if x0 <= 3: x0 = 0
        if y1 >= 15: y1 = 18
        if x1 >= 14: x1 = 17
        if y0 <= 3: y0 = 0
        return dict(x0=x0, x1=x1, y0=y0, y1=y1)

    def inreg(self, c, r):
        R = self.R
        return R['x0'] <= c <= R['x1'] and R['y0'] <= r <= R['y1']

    def path(self, i):
        N = self.N
        p = []
        while N[i]['mv'] is not None:
            p.append(i)
            i = N[i]['par']
        return i, p[::-1]

    def stones(self, i):  # all stones on the board at this node (start position + moves), used to detect stones outside the region
        N = self.N
        ri, p = self.path(i)
        d = dict(self.parse(N[ri]['pos']))
        for j in p:
            d[(N[j]['mv'][0] % 18, N[j]['mv'][0] // 18)] = N[j]['mv'][1]
        return d

    def q(self, i, ctx, qid, visits):
        N = self.N
        ri, p = self.path(i)
        ini = [[v, g(*k)] for k, v in self.parse(N[ri]['pos']).items() if ctx or self.inreg(*k)]
        mv = [[N[j]['mv'][1], g(N[j]['mv'][0] % 18, N[j]['mv'][0] // 18)] for j in p
              if (ctx or self.inreg(N[j]['mv'][0] % 18, N[j]['mv'][0] // 18))]
        allow = self.allow
        return {'id': qid, 'initialStones': ini, 'moves': mv, 'rules': 'japanese', 'komi': 6.5,
                'boardXSize': 19, 'boardYSize': 19, 'analyzeTurns': [len(mv)], 'maxVisits': visits,
                'allowMoves': [{'player': 'B', 'moves': allow, 'untilDepth': 40},
                               {'player': 'W', 'moves': allow, 'untilDepth': 40}]}

    def live(self, i):
        N = self.N
        return [c for c in N[i]['ch'] if not N[c]['dead']]

    def todo(self, limit=0):
        N = self.N
        todo = [i for i in range(len(N)) if not N[i]['dead'] and len(self.live(i)) >= 2]  # branch points only
        rootkids = [i for i in range(len(N)) if not N[i]['dead'] and N[i]['setup'] and len(self.live(i)) == 1 and N[i]['par'] == 0]
        todo = sorted(set(todo + rootkids))
        if limit:
            todo = todo[:limit]
        return todo


# --------------------------------------------------------------------------- KataGo process
class Engine:
    def __init__(self, cmd, err_path):
        self.cmd = cmd
        self.err = open(err_path, 'a', encoding='utf-8')
        self.p = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=self.err,
                                  text=True, encoding='utf-8')
        self.t0 = time.time()
        self.info = {}

    def _readline(self):
        line = self.p.stdout.readline()
        if not line:
            raise RuntimeError('The KataGo process has exited (stdout closed). Check the end of katago_stderr.log; common causes: missing CUDA/cuDNN libraries, out of GPU memory, wrong model file path.')
        return line

    def query_version(self):
        self.p.stdin.write(json.dumps({'id': '__ver', 'action': 'query_version'}) + '\n')
        self.p.stdin.flush()
        d = json.loads(self._readline())
        self.info = {k: d.get(k) for k in ('version', 'git_hash')}
        return self.info

    def run(self, reqs, tag=''):
        """Same as analyze2.run: write all requests at once, collect the responses by id, and convert Black's perspective to the side to move's."""
        for r in reqs:
            self.p.stdin.write(json.dumps(r) + '\n')
        self.p.stdin.flush()
        res = {}
        t0 = time.time()
        for _ in reqs:
            d = json.loads(self._readline())
            if 'moveInfos' not in d:
                res[d['id']] = {'err': str(d)[:120]}
                continue
            t = d['rootInfo']['currentPlayer']
            sg = 1 if t == 'B' else -1
            mi = sorted(d['moveInfos'], key=lambda m: m['order'])
            res[d['id']] = {'turn': t, 'score': round(sg * d['rootInfo']['scoreLead'], 1),
                            'wr': round(d['rootInfo']['winrate'] if t == 'B' else 1 - d['rootInfo']['winrate'], 3),
                            'cand': [{'m': m['move'], 'v': m['visits'], 's': round(sg * m['scoreLead'], 1),
                                      'wr': round(m['winrate'] if t == 'B' else 1 - m['winrate'], 3),
                                      'p': round(m['prior'], 3), 'pv': m['pv'][:8]} for m in mi[:14]]}
            n = len(res)
            if n == len(reqs) or n % 5 == 0:
                print('  %s %d/%d  %.0fs' % (tag, n, len(reqs), time.time() - t0), flush=True)
        return res

    def close(self):
        try:
            self.p.stdin.close()
            self.p.wait(timeout=20)
        except Exception:
            self.p.kill()


# --------------------------------------------------------------------------- full flow for one group (phase 1 + phase 2)
def analyze_group(N, eng, VB=220, VF=100, limit=0):
    G = Group(N)
    todo = G.todo(limit)
    print('  region', G.R, ' branch points', len(todo), flush=True)
    res = {}
    # Phase 1: main search at the branch points (when there are stones outside the region, one more control search without them)
    reqs = []
    for i in todo:
        reqs.append(G.q(i, True, '%d' % i, VB))
        st = G.stones(i)
        if any(not G.inreg(*k) for k in st):
            reqs.append(G.q(i, False, '%d_0' % i, min(VB, 200)))
    res.update(eng.run(reqs, 'phase 1'))
    # Phase 2: for each teacher move the AI search gave no reliable visit count, evaluate it separately and forced
    # (the position after that move, negated because it is the opponent's perspective)
    reqs = []
    need = []
    for i in todo:
        a = res.get('%d' % i)
        if not a or 'cand' not in a:
            continue
        tv = max(m['v'] for m in a['cand'])
        for c in G.live(i):
            if N[c]['mv'] is None:
                continue
            m = g(N[c]['mv'][0] % 18, N[c]['mv'][0] // 18)
            cm = next((x for x in a['cand'] if x['m'] == m), None)
            if cm and cm['v'] >= max(10, .1 * tv):
                continue
            if m not in G.allow:
                continue  # move outside the region (tenuki etc.): not evaluated
            reqs.append(G.q(c, True, 'f%d' % c, VF))
            need.append((i, c, m))
    fr = eng.run(reqs, 'phase 2') if reqs else {}
    for i, c, m in need:
        f = fr.get('f%d' % c)
        a = res['%d' % i]
        if not f or 'cand' not in f or not f['cand']:
            continue
        t = f['cand'][0]
        a['cand'].append({'m': m, 'v': VF, 's': round(-f['score'], 1), 'wr': round(1 - f['wr'], 3), 'p': -1,
                          'pv': [m] + t['pv'][:7], 'f': 1})
    return res, dict(region=G.R, branch_nodes=len(todo), queries=len(res) + len(fr), forced=len(need))


CFG = """# Generated by analyze_remote.py; the search parameters match the local analysis.cfg, only the log directory differs
logDir = {logdir}
reportAnalysisWinratesAs = BLACK
maxVisits = 500
numAnalysisThreads = {nat}
numSearchThreadsPerAnalysisThread = {nst}
nnMaxBatchSize = {bs}
nnCacheSizePowerOfTwo = 23
nnMutexPoolSizePowerOfTwo = 17
nnRandomize = true
"""


def main():
    ap = argparse.ArgumentParser(description='Remote GPU batch version of analyze2.py')
    ap.add_argument('--in', dest='inp', required=True, help='directory containing g_*.n.json, or a single n.json file')
    ap.add_argument('--out', required=True, help='output directory (g_*.a2.json is written here)')
    ap.add_argument('--katago', default=os.environ.get('KATAGO_BIN', './katago'))
    ap.add_argument('--model', default=os.environ.get('KATAGO_MODEL', ''))
    ap.add_argument('--cfg', default='', help='your own analysis.cfg; generated automatically if omitted (must have reportAnalysisWinratesAs=BLACK)')
    ap.add_argument('--engine-cmd', default='', help='(for tests) use this command instead of katago, e.g. "python fake_katago.py"')
    ap.add_argument('--visits', type=int, default=220, help='main search visits (local baseline 220)')
    ap.add_argument('--forced', type=int, default=100, help='forced evaluation visits (local baseline 100)')
    ap.add_argument('--threads', type=int, default=2, help='numAnalysisThreads (can be raised to 3~4 on a strong GPU)')
    ap.add_argument('--search-threads', type=int, default=16)
    ap.add_argument('--batch', type=int, default=64, help='nnMaxBatchSize')
    ap.add_argument('--limit', type=int, default=int(os.environ.get('LIMIT', '0')), help='analyze only the first N branch points per group (trial run)')
    ap.add_argument('--force', action='store_true', help='recompute even if an a2.json already exists')
    ap.add_argument('--only', default='', help='only process groups whose file name contains this substring')
    a = ap.parse_args()

    files = [a.inp] if os.path.isfile(a.inp) else sorted(glob.glob(os.path.join(a.inp, 'g_*.n.json')))
    if a.only:
        files = [f for f in files if a.only in os.path.basename(f)]
    if not files:
        sys.exit('No g_*.n.json found in %s' % a.inp)
    os.makedirs(a.out, exist_ok=True)
    todo_files = []
    for f in files:
        name = os.path.basename(f)[:-len('.n.json')]
        outp = os.path.join(a.out, name + '.a2.json')
        if os.path.exists(outp) and os.path.getsize(outp) > 2 and not a.force:
            print('skip (exists)', outp)
            continue
        todo_files.append((f, name, outp))
    print('%d of %d groups to process' % (len(todo_files), len(files)), flush=True)
    if not todo_files:
        return

    if a.engine_cmd:
        cmd = shlex.split(a.engine_cmd)
    else:
        cfg = a.cfg
        if not cfg:
            cfg = os.path.join(a.out, 'analysis_remote.cfg')
            open(cfg, 'w', encoding='utf-8').write(CFG.format(logdir=os.path.join(a.out, 'katago_logs').replace('\\', '/'),
                                                                nat=a.threads, nst=a.search_threads, bs=a.batch))
        else:
            txt = open(cfg, encoding='utf-8', errors='replace').read()
            if not re.search(r'^\s*reportAnalysisWinratesAs\s*=\s*BLACK\b', txt, re.M):
                print('warning: the supplied cfg has no reportAnalysisWinratesAs = BLACK; it is forced with -override-config.', file=sys.stderr)
        if not a.model or not os.path.exists(a.model):
            sys.exit('Model file not found: %r (pass the .bin.gz with --model)' % a.model)
        cmd = [a.katago, 'analysis', '-model', a.model, '-config', cfg, '-override-config', 'reportAnalysisWinratesAs=BLACK']
    eng = Engine(cmd, os.path.join(a.out, 'katago_stderr.log'))
    try:
        ver = eng.query_version()
        print('KataGo', ver, flush=True)
        T0 = time.time()
        for k, (f, name, outp) in enumerate(todo_files, 1):
            print('[%d/%d] %s' % (k, len(todo_files), name), flush=True)
            N = json.load(open(f, encoding='utf-8'))
            t0 = time.time()
            res, stat = analyze_group(N, eng, a.visits, a.forced, a.limit)
            tmp = outp + '.tmp'
            json.dump(res, open(tmp, 'w'))  # same json.dump defaults as analyze2.py => same format
            os.replace(tmp, outp)
            dt = time.time() - t0
            meta = dict(stat, group=name, seconds=round(dt), visits=a.visits, forced_visits=a.forced, katago=ver,
                        model=os.path.basename(a.model) if a.model else a.engine_cmd, finished=time.strftime('%Y-%m-%d %H:%M:%S'),
                        note='generated by analyze_remote.py (a port of analyze2.py)')
            json.dump(meta, open(outp[:-len('.json')] + '.meta.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
            print('  done %s  took %ds  total %ds' % (name, dt, time.time() - T0), flush=True)
    finally:
        eng.close()
    print('all done ->', os.path.abspath(a.out))


if __name__ == '__main__':
    main()

In [ ]:
%%writefile setup_katago.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Download and check the Linux GPU build of KataGo (works on Colab / Kaggle / Slurm nodes). No root needed, no system libraries are touched.

What it does:
  1) Downloads katago-<ver>-<build>-linux-x64.zip from the GitHub release and unpacks it (default v1.18.2 + cuda12.1-cudnn9.8.0:
     it is built against the lowest CUDA version, so it is the most forward-compatible with the newer CUDA 12.x runtimes in
     cloud images; since v1.18.x, Transformer networks need cuDNN >= 9.8);
  2) uses ldd to find missing shared libraries; if cuDNN/cuBLAS/cudart/nvrtc are missing, installs them with pip into a
     separate directory (--target, system Python packages are untouched) and builds the matching LD_LIBRARY_PATH;
  3) locates the weights file (given with --model, or searched for in the usual places; otherwise --weights-url is required);
  4) writes <dir>/env.sh and <dir>/env.json, and finally prints KATAGO_BIN / KATAGO_MODEL / LD_LIBRARY_PATH.

References (checked 2026-11): KataGo v1.18.2 release page https://github.com/lightvector/KataGo/releases/tag/v1.18.2
  - "Use CUDNN >= 9.8.0. The CUDNN 8.9.7 builds will be a LOT slower when running transformers"
  - "TensorRT versions older than 10 are not supported."
  - v1.18.2 adds tensor-core attention for Turing (RTX 20xx / Tesla T4, sm_75), speeding up Transformer networks by more than 50%.
Note: this has not been run on real hardware (the environment it was written in had no GPU); the version combination was picked
from the official release notes. If something fails, see the "Troubleshooting" section of README.md.
"""
import argparse, glob, json, os, shutil, subprocess, sys, urllib.request, zipfile


def sh(cmd, env=None):
    return subprocess.run(cmd, capture_output=True, text=True, env=env)


def download(url, dst):
    print('Downloading', url, flush=True)
    req = urllib.request.Request(url, headers={'User-Agent': 'josekiapp-remote/1'})
    with urllib.request.urlopen(req) as r, open(dst + '.part', 'wb') as f:
        total = int(r.headers.get('Content-Length', '0') or 0)
        got = 0
        while True:
            b = r.read(1 << 20)
            if not b:
                break
            f.write(b)
            got += len(b)
            if total:
                print('\r  %d / %d MB' % (got >> 20, total >> 20), end='', flush=True)
    print()
    os.replace(dst + '.part', dst)


def missing_libs(binary, env):
    r = sh(['ldd', binary], env)
    return sorted({ln.split('=>')[0].strip() for ln in r.stdout.splitlines() if 'not found' in ln}), r.stdout


def find_model(explicit):
    if explicit:
        return explicit if os.path.exists(explicit) else ''
    pats = ['./kata1-tf3-b11c768*.bin.gz', './*.bin.gz', '/content/*.bin.gz', '/content/drive/MyDrive/josekiapp/*.bin.gz',
            '/content/drive/MyDrive/*.bin.gz', '/kaggle/input/*/*.bin.gz', '/kaggle/input/*/*/*.bin.gz', os.path.expanduser('~/*.bin.gz')]
    for p in pats:
        hit = sorted(glob.glob(p))
        if hit:
            return hit[0]
    return ''


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--dir', default='katago_env')
    ap.add_argument('--ver', default=os.environ.get('KATAGO_VER', 'v1.18.2'))
    ap.add_argument('--build', default=os.environ.get('KATAGO_BUILD', 'cuda12.1-cudnn9.8.0'),
                    help='e.g. cuda12.1-cudnn9.8.0 / cuda12.5-cudnn9.8.0 / cuda12.8-cudnn9.8.0 / cuda13.2-cudnn9.24.0')
    ap.add_argument('--model', default=os.environ.get('KATAGO_MODEL', ''))
    ap.add_argument('--weights-url', default=os.environ.get('WEIGHTS_URL', ''))
    ap.add_argument('--no-pip-libs', action='store_true', help='do not try to pip-install the NVIDIA runtime libraries when libraries are missing')
    a = ap.parse_args()

    d = os.path.abspath(a.dir)
    os.makedirs(d, exist_ok=True)
    nv = sh(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
    print('GPU:', nv.stdout.strip() or '(nvidia-smi unavailable -- no GPU runtime? Colab: Runtime > Change runtime type > GPU)')

    binary = os.path.join(d, 'katago')
    if not os.path.exists(binary):
        name = 'katago-%s-%s-linux-x64.zip' % (a.ver, a.build)
        z = os.path.join(d, name)
        download('https://github.com/lightvector/KataGo/releases/download/%s/%s' % (a.ver, name), z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(d)
        os.chmod(binary, 0o755)
    print('KataGo:', binary)

    env = dict(os.environ)
    miss, out = missing_libs(binary, env)
    libpaths = []
    if miss and not a.no_pip_libs:
        print('Missing shared libraries:', miss)
        nvdir = os.path.join(d, 'nvlibs')
        pk = ['nvidia-cudnn-cu12', 'nvidia-cublas-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cuda-nvrtc-cu12']
        print('Installing with pip into a separate directory %s (system packages are not affected): %s' % (nvdir, ' '.join(pk)), flush=True)
        r = sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', nvdir] + pk)
        if r.returncode:
            print(r.stderr[-800:])
        libpaths = sorted(glob.glob(os.path.join(nvdir, 'nvidia', '*', 'lib')))
        env['LD_LIBRARY_PATH'] = ':'.join(libpaths + [env.get('LD_LIBRARY_PATH', '')]).strip(':')
        miss, out = missing_libs(binary, env)
    if miss:
        # system libraries such as libzip: try apt when running as root (works on Colab, not on clusters)
        if hasattr(os, 'geteuid') and os.geteuid() == 0 and any('zip' in m for m in miss) and shutil.which('apt-get'):
            for pkg in ('libzip4', 'libzip4t64', 'libzip5'):
                if sh(['apt-get', 'install', '-y', '-qq', pkg]).returncode == 0:
                    break
            miss, out = missing_libs(binary, env)
    if miss:
        print('Still missing:', miss, '\n--- ldd ---\n' + out)
        print('=> See "Troubleshooting" in README.md: try another --build (cuda12.5/12.8), use the OpenCL/Eigen build, or ask the cluster admin to load the cuda/cudnn modules.')
    else:
        print('ldd check passed (no missing shared libraries)')

    model = find_model(a.model)
    if not model and a.weights_url:
        model = os.path.join(d, os.path.basename(a.weights_url.split('?')[0]))
        if not os.path.exists(model):
            download(a.weights_url, model)
    if not model:
        print('!! No weights file (*.bin.gz) found. Upload the same kata1-tf3-b11c768-s12752M-d6531M.bin.gz as on the local machine (about 262MB), or set WEIGHTS_URL / --weights-url.')
    else:
        print('Weights:', model, '(%d MB)' % (os.path.getsize(model) >> 20))

    ld = env.get('LD_LIBRARY_PATH', '')
    info = {'katago': binary, 'model': model, 'ld_library_path': ld, 'missing_libs': miss, 'ver': a.ver, 'build': a.build}
    json.dump(info, open(os.path.join(d, 'env.json'), 'w'), indent=1)
    open(os.path.join(d, 'env.sh'), 'w').write('export LD_LIBRARY_PATH="%s"\nexport KATAGO_BIN="%s"\nexport KATAGO_MODEL="%s"\n' % (ld, binary, model))
    print('Wrote %s/env.sh (bash: source it)' % d)
    print(json.dumps(info, ensure_ascii=False))


if __name__ == '__main__':
    main()

## Input and output locations
- **Input**: looks for `g_*.n.json` in `IN_DIR`; if there are none, Kaggle searches `/kaggle/input/*/` automatically and Colab shows an upload box (choose a zip or several json files).
- **Output**: written to `WORK/out` by default; on Colab, set `USE_DRIVE=True` to keep progress across disconnects (a Google Drive authorization prompt appears; you approve it yourself).

In [ ]:
import zipfile, shutil
USE_DRIVE = False            # Colab: True => put input and output under MyDrive/josekiapp/, so a disconnect does not lose them
IN_DIR = os.path.join(WORK, 'in')
OUT_DIR = os.path.join(WORK, 'out')
if ON_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    base = '/content/drive/MyDrive/josekiapp'
    IN_DIR, OUT_DIR = base + '/in', base + '/out'
os.makedirs(IN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

def have_inputs():
    return sorted(glob.glob(IN_DIR + '/g_*.n.json'))

if not have_inputs():
    for d in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        for f in glob.glob(d + '/g_*.n.json'):
            shutil.copy(f, IN_DIR)
if not have_inputs() and ON_COLAB:
    from google.colab import files
    print('Select a zip containing g_*.n.json, or multi-select the json files to upload:')
    for name, data in files.upload().items():
        p = os.path.join(IN_DIR, name)
        open(p, 'wb').write(data)
        if name.endswith('.zip'):
            with zipfile.ZipFile(p) as z:
                for m in z.namelist():
                    if m.endswith('.n.json'):
                        open(os.path.join(IN_DIR, os.path.basename(m)), 'wb').write(z.read(m))
inp = have_inputs()
print('%d input groups:' % len(inp)); [print('  ', os.path.basename(f)) for f in inp]
assert inp, 'No g_*.n.json found -- see the instructions above'

## Install KataGo (GPU build) and locate the weights
The weights are found automatically if they are in one of these places: the working directory, `/content/*.bin.gz`, `MyDrive/josekiapp/*.bin.gz`, `/kaggle/input/*/`.
You can also set `WEIGHTS_PATH` (a local path) or `WEIGHTS_URL` (a download URL). **Use the same weights as the local machine**, otherwise old and new analyses are not comparable.

In [ ]:
WEIGHTS_PATH = ''   # e.g. '/content/drive/MyDrive/josekiapp/kata1-tf3-b11c768-s12752M-d6531M.bin.gz'
WEIGHTS_URL = ''    # e.g. a direct download link to that weights file (leave empty if unsure and upload it instead)
KATAGO_BUILD = 'cuda12.1-cudnn9.8.0'   # if libraries are missing, try cuda12.5-cudnn9.8.0 / cuda12.8-cudnn9.8.0 in turn
cmd = [sys.executable, '-u', 'setup_katago.py', '--dir', WORK + '/katago_env', '--build', KATAGO_BUILD]
if WEIGHTS_PATH: cmd += ['--model', WEIGHTS_PATH]
if WEIGHTS_URL: cmd += ['--weights-url', WEIGHTS_URL]
subprocess.run(cmd, check=True)
ENV = json.load(open(WORK + '/katago_env/env.json'))
if not ENV['model'] and ON_COLAB:
    from google.colab import files
    print('Upload the weights file kata1-tf3-b11c768-s12752M-d6531M.bin.gz (about 262MB, the upload is slow):')
    for name, data in files.upload().items():
        open(WORK + '/' + name, 'wb').write(data)
    subprocess.run(cmd, check=True)
    ENV = json.load(open(WORK + '/katago_env/env.json'))
assert ENV['model'], 'no weights file'
assert not ENV['missing_libs'], 'missing shared libraries: %s (see Troubleshooting in the README)' % ENV['missing_libs']
os.environ['LD_LIBRARY_PATH'] = ENV['ld_library_path']
print('Ready:', ENV['katago'], ENV['model'])

## (Optional) Trial run on 3 branch points to check the speed
The local integrated GPU takes about 11 seconds per query (220 visits). The time of each query is printed here, so you can estimate how long the whole batch will take.

In [ ]:
t0 = time.time()
subprocess.run([sys.executable, '-u', 'analyze_remote.py', '--in', inp[0], '--out', WORK + '/trial', '--katago', ENV['katago'],
                '--model', ENV['model'], '--limit', '3', '--force'], check=True)
print('Trial run took %.0f seconds (including model loading and first-time compilation, so the first run is slower)' % (time.time() - t0))

## Full batch run (can be interrupted at any time; re-running resumes automatically)

In [ ]:
r = subprocess.run([sys.executable, '-u', 'analyze_remote.py', '--in', IN_DIR, '--out', OUT_DIR, '--katago', ENV['katago'],
                    '--model', ENV['model'], '--visits', '220', '--forced', '100'])
print('exit code', r.returncode)

## Package the results

In [ ]:
res = sorted(glob.glob(OUT_DIR + '/g_*.a2.json')) + sorted(glob.glob(OUT_DIR + '/g_*.a2.meta.json'))
zp = WORK + '/a2_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in res: z.write(f, os.path.basename(f))
print('%d files -> %s' % (len(res), zp))
if ON_COLAB:
    from google.colab import files
    files.download(zp)